# Table Spend

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re
Spend = pd.read_excel('/content/drive/MyDrive/fin_project/not clean/Spend.xlsx')
print(Spend.shape)
print()
summary_spend = pd.DataFrame({
    'dtype': Spend.dtypes,
    'non_null': Spend.notna().sum(),
    'null': Spend.isna().sum(),
    'null_%': (Spend.isna().sum() / len(Spend) * 100).round(1),
    'unique': Spend.nunique()
})
print(summary_spend)

In [ ]:
print("Source:")
print(Spend['Source'].value_counts(dropna=False))
print()
print("Spend - basic statistics:")
print(Spend['Spend'].describe())
print(f"\nZero Spend: {(Spend['Spend'] == 0).sum()}")
print(f"Negative Spend: {(Spend['Spend'] < 0).sum()}")
print()
# Are AdGroup and Ad missing values linked?
print(f"AdGroup NaN == Ad NaN: {(Spend['AdGroup'].isna() == Spend['Ad'].isna()).all()}")

In [ ]:
# Zero Spend by Source - which channels have no spend?
print("Share of zero Spend by Source:")
zero_spend = Spend.groupby('Source').apply(
    lambda x: (x['Spend'] == 0).mean()
).sort_values(ascending=False)
print(zero_spend)
print()
# Missing Campaign values by Source
print("Share of missing Campaign by Source:")
camp_null = Spend.groupby('Source').apply(
    lambda x: x['Campaign'].isna().mean()
).sort_values(ascending=False)
print(camp_null)

# Channel type

In [ ]:
# Fix the Warning - add include_groups=False
zero_spend = Spend.groupby('Source')['Spend'].apply(
    lambda x: (x == 0).mean()
).sort_values(ascending=False)

# Create a channel type column
paid_channels = ['Facebook Ads', 'Tiktok Ads', 'Youtube Ads', 'Google Ads']
mixed_channels = ['Bloggers', 'SMM', 'Telegram posts', 'Webinar', 'Test', 'Radio']
free_channels = ['CRM', 'Organic', 'Offline', 'Partnership']

def classify_channel(source):
    if source in paid_channels:
        return 'Paid'
    elif source in mixed_channels:
        return 'Mixed'
    else:
        return 'Free'

Spend['Channel_type'] = Spend['Source'].apply(classify_channel)

print(Spend['Channel_type'].value_counts())
print()
print("Average Spend by channel type:")
print(Spend.groupby('Channel_type')['Spend'].mean().round(2))

In [ ]:
print("Average Spend for each Source (non-zero only):")
print(Spend[Spend['Spend'] > 0].groupby('Source')['Spend'].agg(['mean', 'sum', 'count']).round(2).sort_values('sum', ascending=False))

# Campaign

In [ ]:
print(Spend['Campaign'].value_counts(dropna=False).head(30))

In [ ]:
# Extract the country from the campaign name
def extract_country(campaign):
    if pd.isna(campaign):
        return 'Unknown'
    campaign = str(campaign).upper()
    if '_DE' in campaign or campaign.endswith('DE'):
        return 'DE'
    elif '_PL' in campaign:
        return 'PL'
    elif '_AT' in campaign:
        return 'AT'
    elif '_ENG' in campaign or 'ENG' in campaign:
        return 'ENG'
    return 'Unknown'

# Extract the audience type
def extract_audience(campaign):
    if pd.isna(campaign):
        return 'Unknown'
    campaign = str(campaign).lower()
    if 'wide' in campaign:
        return 'Wide'
    elif 'retargeting' in campaign:
        return 'Retargeting'
    elif 'lal' in campaign:
        return 'Lookalike'
    elif 'interests' in campaign:
        return 'Interests'
    elif 'women' in campaign:
        return 'Women'
    elif 'recentlymoved' in campaign:
        return 'Recently Moved'
    elif 'webinar' in campaign:
        return 'Webinar'
    elif 'brand' in campaign:
        return 'Brand'
    elif 'performance' in campaign:
        return 'Performance Max'
    return 'Other'

Spend['Campaign_country'] = Spend['Campaign'].apply(extract_country)
Spend['Campaign_audience'] = Spend['Campaign'].apply(extract_audience)

print("Distribution of campaign countries:")
print(Spend['Campaign_country'].value_counts())
print()
print("Distribution of audience type:")
print(Spend['Campaign_audience'].value_counts())

# Impressions + Product Analytics

In [ ]:
print(Spend['Impressions'].describe())
print(f"Zero values: {(Spend['Impressions'] == 0).sum()}")

In [ ]:
print(Spend.loc[Spend['Impressions'] == 0, 'Source'].value_counts())

In [ ]:
mask_zero_imp = Spend['Impressions'] == 0
print(f"Zero Impressions + non-zero Clicks: {(mask_zero_imp & (Spend['Clicks'] > 0)).sum()}")
print(f"Zero Impressions + non-zero Spend: {(mask_zero_imp & (Spend['Spend'] > 0)).sum()}")

In [ ]:
# 1370 rows with clicks but no impressions
mask_clicks_no_imp = (Spend['Impressions'] == 0) & (Spend['Clicks'] > 0)
print("Clicks without impressions by Source:")
print(Spend.loc[mask_clicks_no_imp, 'Source'].value_counts())
print()
print("Click statistics for these rows:")
print(Spend.loc[mask_clicks_no_imp, 'Clicks'].describe())

In [ ]:
mask_spend_no_imp = (Spend['Impressions'] == 0) & (Spend['Spend'] > 0)
print("Spend without impressions by Source:")
print(Spend.loc[mask_spend_no_imp, ['Source', 'Spend', 'Clicks']].groupby('Source').agg({'Spend': 'sum', 'Clicks': 'sum'}))

In [ ]:
# CTR - only where impressions exist
Spend['CTR'] = (Spend['Clicks'] / Spend['Impressions']).replace([float('inf'), float('nan')], None)

# CPM - cost per 1000 impressions (only where impressions and spend exist)
Spend['CPM'] = (Spend['Spend'] / Spend['Impressions'] * 1000).replace([float('inf'), float('nan')], None)

# CPC - cost per click (only where clicks and spend exist)
Spend['CPC'] = (Spend['Spend'] / Spend['Clicks']).replace([float('inf'), float('nan')], None)

# Average metrics for paid channels
paid_mask = Spend['Channel_type'] == 'Paid'
print("Average metrics for paid channels:")
print(f"CTR: {Spend.loc[paid_mask, 'CTR'].mean():.4f} ({Spend.loc[paid_mask, 'CTR'].mean()*100:.2f}%)")
print(f"CPM: {Spend.loc[paid_mask, 'CPM'].mean():.2f}€")
print(f"CPC: {Spend.loc[paid_mask, 'CPC'].mean():.2f}€")
print()
print("CTR, CPM, CPC by Source (paid channels):")
print(Spend.loc[paid_mask].groupby('Source')[['CTR', 'CPM', 'CPC']].mean().round(4))

**CTR (click-through rate):**

Google Ads — leader (6.4%), 4x above average — the most relevant audience (search queries)
Facebook Ads (1.6%) — average
Youtube Ads and Tiktok Ads — low CTR (0.6-0.7%), normal for video ads

**CPM (cost per 1000 impressions):**

Google Ads — most expensive (91.98€) — search ads are always pricier
Facebook Ads (13.25€) — average
Tiktok Ads and Youtube Ads (3.1-3.3€) — cheapest impressions

**CPC (cost per click):**

Youtube Ads (0.45€) — cheapest click
Tiktok Ads (0.59€) — second cheapest
Facebook Ads (0.92€) — most expensive click

**Main takeaway for product analytics:**
Google Ads delivers the highest CTR but the most expensive CPM — it targets a hot, high-intent audience. Tiktok/Youtube attract a broad audience cheaply. This matters for CAC calculations — a cheap click doesn't always mean a cheap lead.

# Clicks

In [ ]:
print(Spend['Clicks'].describe())
print(f"Zero values: {(Spend['Clicks'] == 0).sum()}")

In [ ]:
mask_zero_clicks = Spend['Clicks'] == 0
print("Zero clicks by Source:")
print(Spend.loc[mask_zero_clicks, 'Source'].value_counts())
print()
print("Zero clicks + non-zero Spend:")
print((mask_zero_clicks & (Spend['Spend'] > 0)).sum())
print()
print("Zero clicks + non-zero Impressions:")
print((mask_zero_clicks & (Spend['Impressions'] > 0)).sum())

In [ ]:
mask_zero_clicks_spend = (Spend['Clicks'] == 0) & (Spend['Spend'] > 0)
print("Spend without clicks by Source:")
print(Spend.loc[mask_zero_clicks_spend].groupby('Source')['Spend'].agg(['sum', 'count']).sort_values('sum', ascending=False).round(2))
print()
total_wasted = Spend.loc[mask_zero_clicks_spend, 'Spend'].sum()
total_spend = Spend['Spend'].sum()
print(f"Total spent with no clicks: {total_wasted:.2f}€ ({total_wasted/total_spend*100:.1f}% of total budget)")

# CPM

In [ ]:
Spend['CPM'] = pd.to_numeric(Spend['CPM'], errors='coerce')
Spend['CPC'] = pd.to_numeric(Spend['CPC'], errors='coerce')
Spend['CTR'] = pd.to_numeric(Spend['CTR'], errors='coerce')

print(Spend[['CTR', 'CPM', 'CPC']].dtypes)

# Aggregated Table

In [ ]:
PATH = '/content/drive/MyDrive/fin_project/'
Spend = pd.read_parquet(PATH + 'Spend_clean.parquet')

# Aggregate Spend to campaign level
Spend_agg = Spend.groupby('Campaign').agg(
    Total_Spend    = ('Spend',       'sum'),
    Total_Impressions = ('Impressions', 'sum'),
    Total_Clicks   = ('Clicks',      'sum'),
    Source         = ('Source',      'first'),
    Channel_type   = ('Channel_type', 'first'),
    Campaign_country = ('Campaign_country', 'first'),
    Campaign_audience = ('Campaign_audience', 'first'),
).reset_index()

# Compute derived metrics
Spend_agg['CTR'] = (Spend_agg['Total_Clicks'] / Spend_agg['Total_Impressions']).round(4)
Spend_agg['CPM'] = (Spend_agg['Total_Spend'] / Spend_agg['Total_Impressions'] * 1000).round(2)
Spend_agg['CPC'] = (Spend_agg['Total_Spend'] / Spend_agg['Total_Clicks']).round(2)

# Replace inf and NaN (division by 0)
Spend_agg = Spend_agg.replace([float('inf'), float('nan')], 0)

print(f"Spend_agg: {Spend_agg.shape}")
print(Spend_agg.head(10))

# Save
Spend_agg.to_parquet(PATH + 'Spend_agg.parquet', index=False)

print("Saved!")

# Saving

In [ ]:
# Look at duplicates in Spend
dupes = Spend[Spend.duplicated(keep=False)].sort_values(['Date', 'Source', 'Campaign'])
print(f"Rows involved in duplication: {len(dupes)}")
print()
print(dupes.head(10))

In [ ]:
# Check duplicates in Spend
print(f"Duplicates before removal: {Spend.duplicated().sum()}")

# Remove duplicates (917 empty Bloggers rows with zero values)
Spend = Spend.drop_duplicates()
print(f"Duplicates after removal: {Spend.duplicated().sum()}")
print(f"Rows remaining: {len(Spend)}")

In [ ]:
Spend.to_parquet('/content/drive/MyDrive/fin_project/Spend_clean.parquet', index=False)
print("Spend saved!")

# Descriptive Statistics

In [ ]:
# Numeric columns in Spend
numeric_cols_spend = ['Impressions', 'Spend', 'Clicks', 'CTR', 'CPM', 'CPC']

for col in numeric_cols_spend:
    print(f"{'='*40}")
    print(f"Column: {col}")
    print(f"  Mean:    {Spend[col].mean():.4f}")
    print(f"  Median:  {Spend[col].median():.4f}")
    print(f"  Mode:    {Spend[col].mode()[0]:.4f}")
    print(f"  Min:     {Spend[col].min():.4f}")
    print(f"  Max:     {Spend[col].max():.4f}")
    print(f"  Range:   {(Spend[col].max() - Spend[col].min()):.4f}")
    print(f"  Missing: {Spend[col].isna().sum()}")

In [ ]:
# Anomalous CPM
print("Top-5 most expensive CPM:")
print(Spend.nlargest(5, 'CPM')[['Source', 'Campaign', 'Impressions', 'Spend', 'CPM']])
print()
# Anomalous CPC
print("Top-5 most expensive CPC:")
print(Spend.nlargest(5, 'CPC')[['Source', 'Campaign', 'Clicks', 'Spend', 'CPC']])

The maximum CPM (1558€) was recorded for Google Ads search campaigns with narrow targeting (62 impressions) — this is normal for highly competitive keywords. The maximum CPC (26.7€) is for Bloggers — a consequence of the flat-fee placement model (fixed amount / few clicks), not a real auction click cost.

In [ ]:
# Categorical columns in Spend
cat_cols_spend = ['Source', 'Channel_type', 'Campaign_country', 'Campaign_audience']

for col in cat_cols_spend:
    print(f"{'='*40}")
    print(f"Column: {col}")
    print(Spend[col].value_counts(dropna=False).head(10))